In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("games_clean.csv")

print(df.shape)
df.head()

(21925, 46)


,BGGId,Name,YearPublished,GameWeight,AvgRating,BayesAvgRating,StdDev,MinPlayers,MaxPlayers,ComAgeRec,...,Rank:partygames,Rank:childrensgames,Cat:Thematic,Cat:Strategy,Cat:War,Cat:Family,Cat:CGS,Cat:Abstract,Cat:Party,Cat:Childrens
0,1,Die Macher,1986,4.3206,7.61428,7.10363,1.57979,3,5,14.366667,...,21926,21926,0,1,0,0,0,0,0,0
1,2,Dragonmaster,1981,1.9630,6.64537,5.78447,1.45440,3,4,NaN,...,21926,21926,0,1,0,0,0,0,0,0
2,3,Samurai,1998,2.4859,7.45601,7.23994,1.18227,2,4,9.307692,...,21926,21926,0,1,0,0,0,0,0,0
3,4,Tal der Könige,1992,2.6667,6.60006,5.67954,1.23129,2,4,13.000000,...,21926,21926,0,0,0,0,0,0,0,0
4,5,Acquire,1964,2.5031,7.33861,7.14189,1.33583,2,6,11.410256,...,21926,21926,0,1,0,0,0,0,0,0


# **1️- Categorías: ¿Qué tipo de juegos funcionan mejor?**

In [ ]:
categories = [
    "Cat:Thematic",
    "Cat:Strategy",
    "Cat:War",
    "Cat:Family",
    "Cat:CGS",
    "Cat:Abstract",
    "Cat:Party",
    "Cat:Childrens"
]

results = []

for cat in categories:

    results.append({
        "Category": cat.replace("Cat:", ""),
        "AvgRating": df[df[cat] == 1]["BayesAvgRating"].mean(),
        "NumGames": len(df[df[cat] == 1])
    })

category_df = pd.DataFrame(results)

category_df.sort_values(
    "AvgRating",
    ascending=False
)

,Category,AvgRating,NumGames
1,Strategy,6.163389,2319
0,Thematic,6.020867,1224
3,Family,5.903091,2316
6,Party,5.825439,640
4,CGS,5.803606,303
5,Abstract,5.657584,1115
2,War,5.640214,3530
7,Childrens,5.514134,881


¿Qué categorías reciben mejores valoraciones?

In [ ]:
fig = px.bar(
    category_df.sort_values("AvgRating"),
    x="AvgRating",
    y="Category",
    orientation="h",
    text_auto=".2f",
    title="Average Rating by Category"
)

fig.show()

# **2- Complejidad: ¿Los juegos complejos gustan más?**

¿Existe relación entre complejidad y valoración?



In [ ]:
fig = px.scatter(
    df,
    x="GameWeight",
    y="BayesAvgRating",
    opacity=0.4,
    title="Complexity vs Rating"
)

fig.show()

# **3- Kickstarter: ¿Financiarse por crowdfunding es una ventaja?**

¿Los juegos financiados en Kickstarter reciben mejores valoraciones?

In [ ]:
kickstarter_metrics = (
    df.groupby("Kickstarted")
      .agg({
          "BayesAvgRating":"mean",
          "NumOwned":"mean",
          "NumWish":"mean"
      })
      .reset_index()
)

kickstarter_metrics

,Kickstarted,BayesAvgRating,NumOwned,NumWish
0,0,5.670306,1409.877175,202.791682
1,1,5.770518,1787.930101,370.165973


In [ ]:
kickstarter_avg = (
    df.groupby("Kickstarted")["BayesAvgRating"]
      .mean()
      .reset_index()
)

kickstarter_avg["Kickstarted"] = (
    kickstarter_avg["Kickstarted"]
    .map({
        0: "No",
        1: "Yes"
    })
)

fig = px.bar(
    kickstarter_avg,
    x="Kickstarted",
    y="BayesAvgRating",
    text_auto=".2f",
    title="Average Rating by Funding Type"
)

fig.show()

# **4- Segmentos de jugadores**

¿Qué tipo de juegos según la cantidad de jugadores obtienen mejores valoraciones?

In [ ]:
def classify_players(x):
    if x <= 2:
        return "2 Players"
    elif x <= 5:
        return "3-5 Players"
    elif x <= 10:
        return "6-10 Players"
    else:
        return "Large Groups"

df["PlayerSegment"] = df["MaxPlayers"].apply(classify_players)

In [ ]:
df["PlayerSegment"].value_counts()

,count
PlayerSegment,
3-5 Players,10199
6-10 Players,6085
2 Players,4890
Large Groups,751


In [ ]:
segment_df = (
    df.groupby("PlayerSegment")["BayesAvgRating"]
      .agg(["mean","median","count","std"])
      .reset_index()
)

segment_df

,PlayerSegment,mean,median,count,std
0,2 Players,5.640381,5.548165,4890,0.278954
1,3-5 Players,5.741322,5.560320,10199,0.425226
2,6-10 Players,5.638834,5.531950,6085,0.310666
3,Large Groups,5.604336,5.527700,751,0.268991


El número máximo de jugadores no parece ser un factor determinante en la valoración de un juego.

# **5- Evolución temporal**

¿Está creciendo el mercado?

In [ ]:
games_per_year = (
    df[df["YearPublished"] >= 1950]
    .groupby("YearPublished")
    .size()
    .reset_index(name="Games")
)

fig = px.line(
    games_per_year,
    x="YearPublished",
    y="Games",
    title="Games Published per Year (1950-Present)"
)

fig.show()

¿Los juegos modernos están mejor valorados? Rating medio por década.

In [ ]:
df_1950 = df[df["YearPublished"] >= 1950].copy()

df_1950["Decade"] = (
    (df_1950["YearPublished"] // 10) * 10
)

rating_decade = (
    df_1950.groupby("Decade")["BayesAvgRating"]
    .mean()
    .reset_index()
)

fig = px.line(
    rating_decade,
    x="Decade",
    y="BayesAvgRating",
    markers=True,
    title="Average Rating by Decade"
)

fig.show()

¿Los juegos modernos reciben mejores notas, los juegos antiguos están infrarrepresentados?

In [ ]:
rating_period = (
    df.groupby(
        pd.cut(
            df["YearPublished"],
            bins=[1950,1970,1990,2010,2022]
        )
    )["BayesAvgRating"]
    .mean()
    .reset_index()
)

rating_period

/tmp/ipykernel_2685/1764134020.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,YearPublished,BayesAvgRating
0,"(1950, 1970]",5.514080
1,"(1970, 1990]",5.573315
2,"(1990, 2010]",5.650644
3,"(2010, 2022]",5.739698


# **6- Popularidad**

¿Qué juegos dominan el mercado?

In [ ]:
top_owned = (
    df.nlargest(
        20,
        "NumOwned"
    )[
        [
            "Name",
            "NumOwned"
        ]
    ]
)

fig = px.bar(
    top_owned,
    x="NumOwned",
    y="Name",
    orientation="h",
    title="Most Owned Games"
)

fig.show()

# **7- Factores de éxito**

Ahora vamos a identificar qué factores están más relacionados con el éxito y la valoración de los juegos.

**7.1 Características del producto**

Complejidad vs valoración

Existe una relación positiva moderada (≈0.29). Los juegos más complejos tienden a recibir mejores valoraciones, aunque la complejidad no explica por sí sola el éxito.

Duración vs valoración

La duración apenas muestra relación con la valoración (≈0.02). Los juegos largos no son necesariamente mejor valorados que los cortos.

**7.2 Categorías de juego**

Valoración por categorías

Los juegos de estrategia y temática reciben las valoraciones más altas, mientras que los juegos infantiles obtienen las puntuaciones más bajas.

**7.3 Recepción del mercado**

Aquí ya no hablamos de características del producto.

Ahora analizamos cómo responde la comunidad.

In [ ]:
corr_rating = (
    df[
        [
            "BayesAvgRating",
            "NumOwned",
            "NumUserRatings",
            "NumWish"
        ]
    ]
    .corr()["BayesAvgRating"]
    .drop("BayesAvgRating")
    .sort_values(ascending=False)
)

print(corr_rating)

NumWish           0.774962
NumOwned          0.638548
NumUserRatings    0.631087
Name: BayesAvgRating, dtype: float64


Los juegos más valorados suelen ser también los más deseados, poseídos y comentados por la comunidad. Hay una diferencia enorme entre:

*NumOwned:*	Gente que posee el juego

*NumUserRatings:*	Gente que lo ha jugado y valorado

*NumWish:*	Gente que quiere tenerlo

Entonces, ¿por qué correlaciona tanto con el rating NumWish?

Porque probablemente ocurre algo como esto:

* Los juegos muy bien valorados generan interés.
* Los juegos muy esperados reciben más atención.
* Los juegos populares entran en más listas de deseos.

Pero aquí aparece un problema:

La correlación NO significa causalidad:

"Estar en muchas wishlists hace que el juego sea bueno"

Podría ser justo al revés:

"Los juegos buenos acaban en muchas wishlists"

# **8- Sesgos y gobernanza**

¿Hasta qué punto los resultados obtenidos representan realmente el mercado de juegos de mesa?

**1— Sesgo de comunidad**

Categoría	-> Rating

Strategy -> 6.16

Thematic ->	6.02

Childrens ->	5.51

Pregunta:

*¿Son realmente peores los juegos infantiles?*

Probablemente no.

Lo que ocurre es que BoardGameGeek está compuesto mayoritariamente por:

* Jugadores adultos
* Aficionados frecuentes
* Gente interesada en juegos más complejos

Por tanto, el dataset puede sobrevalorar juegos de estrategia y temática frente a juegos infantiles o familiares debido al perfil de los usuarios que participan en la plataforma.

**2— Sesgo de popularidad**

YNumWish           0.774962
NumOwned          0.638548
NumUserRatings    0.631087

Pregunta:

*¿Los juegos reciben mejores valoraciones porque son mejores o porque son más conocidos?*

No podemos saberlo.

Pero sí sabemos que tienen mejores ratings:

* Juegos más poseídos
* Juegos más votados

Los juegos más populares reciben más atención por parte de la comunidad, lo que puede influir en sus valoraciones y favorecer a títulos ya consolidados.

**3— Sesgo temporal**

Los juegos más recientes reciben mejores valoraciones.

Pero aquí hay dos explicaciones posibles:

*Explicación A:* Los juegos modernos son mejores

La industria ha evolucionado:

* Mejores mecánicas
* Más playtesting
* Más experiencia de diseño
* Mayor profesionalización

Por tanto, los juegos modernos podrían ser realmente mejores.

*Explicación B:* Sesgo de supervivencia

Los juegos de 1955 ¿Cuáles siguen apareciendo hoy en BoardGameGeek?

Probablemente los más conocidos, populares o publicitados

Mientras que miles de juegos con menor inversión o menos populares o con temáticas que no se ajustaban a las tendencias de la época han desaparecido.

Eso hace que los periodos antiguos estén filtrados naturalmente.

Por tanto, la diferencia puede estar influida por un sesgo de supervivencia, ya que muchos títulos antiguos con escasa relevancia comercial han desaparecido del mercado o reciben poca atención dentro de la comunidad actual.

**4— Datos faltantes**

*Family*

Más de 15.000 nulos.

Esto significa que muchos juegos no pertenecen a una familia/franquicia o esa información no fue registrada.

Por tanto, no sería recomendable sacar conclusiones sobre franquicias o familias de juegos usando este dataset.

*LanguageEase*

Casi 6.000 nulos.

Además esta variable depende de aportaciones de la comunidad.

Por tanto, los análisis relacionados con accesibilidad lingüística podrían no representar adecuadamente todos los juegos.

*ComAgeRec*

Más de 5.500 nulos.

La edad recomendada por la comunidad no está disponible para todos los juegos.

Por tanto, las comparaciones entre la edad recomendada por fabricantes y comunidad deben interpretarse con cautela, al no presentarse en todos los juegos y estar basado en un número desconocido de opiniones.

In [ ]:
# A. Top 15 más poseídos

top_owned = (
    df.nlargest(
        15,
        "NumOwned"
    )[
        [
            "Name",
            "NumOwned",
            "BayesAvgRating"
        ]
    ]
)

top_owned

,Name,NumOwned,BayesAvgRating
7919,Pandemic,166497,7.48919
12,Catan,165651,6.97148
673,Carcassonne,159709,7.30890
9861,7 Wonders,119235,7.63557
14811,Codenames,117959,7.51089
14450,7 Wonders Duel,108774,7.98227
8459,Dominion,106256,7.49999
4702,Ticket to Ride,104472,7.30689
14059,Terraforming Mars,99827,8.27421
11807,Love Letter,97734,7.14107


In [ ]:
# Top 15 más poseídos

top_rated = (
    df.nlargest(
        15,
        "BayesAvgRating"
    )[
        [
            "Name",
            "BayesAvgRating",
            "NumOwned"
        ]
    ]
)

top_rated

,Name,BayesAvgRating,NumOwned
14509,Gloomhaven,8.51488,76471
13702,Pandemic Legacy: Season 1,8.44451,69952
17329,Brass: Birmingham,8.41573,36959
14059,Terraforming Mars,8.27421,99827
17834,Twilight Imperium: Fourth Edition,8.25955,20038
20687,Gloomhaven: Jaws of the Lion,8.25163,36046
17127,Gaia Project,8.17758,23772
15360,Star Wars: Rebellion,8.17121,38785
15046,Through the Ages: A New Story of Civilization,8.15369,29759
11220,War of the Ring: Second Edition,8.13104,24622


# **¿Qué significa esto?**

Parece que existen dos tipos de éxito:

**Éxito comercial**

Muchísima gente los posee.

Ejemplos:

Catan
Ticket to Ride
Codenames

**Éxito crítico**

Los aficionados más experimentados los valoran extraordinariamente bien.

Ejemplos:

Gloomhaven
Brass Birmingham
Twilight Imperium
Y esto conecta con algo que vimos antes

Recordemos:

Categorías mejor valoradas
* Strategy
* Thematic
* Complejidad

Correlación:

0.29

Los juegos complejos reciben mejores valoraciones.

Por tanto parece que:

**Mercado masivo**

Prefiere:

* Accesibilidad
* Reglas sencillas
* Partidas rápidas

**Comunidad experta**

Prefiere:

* Estrategia profunda
* Complejidad
* Campañas largas
